# PAP Emulation - SciPy Solver Validation

Copyright © 1993-2026 Abhishek Choudhary. All rights reserved. AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary. Licensed under GNU GPL v3.0. Third-party dependencies retain their own licenses.

Linear state-space execution with six SciPy integration methods, analytical comparison and a cryptographically bound PAP job/result exchange.

**Author:** Abhishek Choudhary · **Affiliation:** AyeAI · **ORCID:** [0009-0002-0684-8320](https://orcid.org/0009-0002-0684-8320)

**Deployment:** browser-only platform; this notebook runs in a user-controlled Colab runtime. GPU/TPU availability is variable and not guaranteed. **Evidence boundary:** results remain `SIMULATED` and `UNVERIFIED`; this is not HIL or physical verification.

## 0. Runtime, security and provenance
Run cells in order. The notebook records backend versions, solver settings, runtime device, model configuration, timestamps and SHA-256 job/result digests. Results must be imported through the separate PAP Colab Bridge. Never upload untrusted native FMUs or binaries.

In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary. All rights reserved.
# AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary.
# SPDX-License-Identifier: GPL-3.0-only
# Embedded in each notebook so notebooks remain independently runnable.
PAP_BRIDGE_VERSION = '1.0.0'
PAP_JOB_SCHEMA = 'pap.emulation.job/1.0'
PAP_RESULT_SCHEMA = 'pap.emulation.result/1.0'

def pap_canonical(value):
    import json
    return json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False, allow_nan=False)

def pap_digest(text):
    import hashlib
    return 'sha256:' + hashlib.sha256(text.encode('utf-8')).hexdigest()

def pap_make_job(backend_id, operation, inputs):
    from datetime import datetime, timezone
    body = {
        'schema': PAP_JOB_SCHEMA,
        'bridgeVersion': PAP_BRIDGE_VERSION,
        'createdAt': datetime.now(timezone.utc).isoformat(),
        'target': {'backendId': backend_id, 'backendVersion': None, 'operation': operation, 'requestedDevice': 'auto'},
        'sir': {'header': {'id': 'colab-reference-fixture', 'irVersion': '1.0.0'}, 'components': []},
        'inputs': inputs,
        'observations': None,
        'configuration': {},
        'provenance': {'source': 'COLAB_REFERENCE_FIXTURE', 'sourceVersion': '1.0.0', 'license': 'GPL-3.0-only'}
    }
    canonical = pap_canonical(body)
    digest = pap_digest(canonical)
    return {**body, 'jobId': 'papjob-' + digest[7:27], 'digest': digest, 'canonicalPayload': canonical}

def pap_verify_job(job, expected_backend):
    import json
    if not isinstance(job, dict) or job.get('schema') != PAP_JOB_SCHEMA:
        raise ValueError('Unsupported/missing PAP job schema')
    payload = job.get('canonicalPayload')
    if not isinstance(payload, str): raise ValueError('Job canonicalPayload missing')
    parsed = json.loads(payload)
    body = {k:v for k,v in job.items() if k not in ('jobId','digest','canonicalPayload')}
    if pap_canonical(parsed) != pap_canonical(body): raise ValueError('Job payload/body mismatch')
    digest = pap_digest(payload)
    if digest != job.get('digest') or job.get('jobId') != 'papjob-' + digest[7:27]:
        raise ValueError('Job digest/jobId verification failed')
    if job.get('target',{}).get('backendId') != expected_backend:
        raise ValueError('Job targets a different backend')
    if not isinstance(job.get('inputs'), dict): raise ValueError('Job inputs must be an object')
    return job

def pap_get_job(backend_id, operation, default_inputs, use_demo=True):
    if use_demo:
        return pap_make_job(backend_id, operation, default_inputs), True
    from google.colab import files
    import json
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.lower().endswith('.json')]
    if len(candidates) != 1: raise ValueError('Upload exactly one browser-exported PAP job .json')
    return pap_verify_job(json.loads(uploaded[candidates[0]].decode('utf-8')), backend_id), False

def pap_make_result(job, backend_id, backend_version, outputs, diagnostics, device='cpu'):
    from datetime import datetime, timezone
    body = {
        'schema': PAP_RESULT_SCHEMA,
        'bridgeVersion': PAP_BRIDGE_VERSION,
        'jobId': job['jobId'],
        'jobDigest': job['digest'],
        'backend': {'id': backend_id, 'version': str(backend_version)},
        'execution': {'status': 'EXECUTED', 'runtime': 'COLAB', 'startedAt': None,
                      'finishedAt': datetime.now(timezone.utc).isoformat(), 'device': str(device)},
        'outputs': outputs,
        'diagnostics': diagnostics,
        'fidelity': 'SIMULATED',
        'physicalVerification': False,
        'evidenceState': 'UNVERIFIED',
        'claims': {'verified': False, 'hil': False, 'physical': False}
    }
    canonical = pap_canonical(body)
    return {**body, 'canonicalPayload': canonical, 'resultDigest': pap_digest(canonical)}

def pap_export(job, result, is_demo, result_filename='pap-emulation-result.json'):
    import json
    with open(result_filename, 'w', encoding='utf-8') as stream:
        json.dump(result, stream, indent=2, ensure_ascii=False, allow_nan=False)
    if is_demo:
        with open('pap-emulation-job.json', 'w', encoding='utf-8') as stream:
            json.dump(job, stream, indent=2, ensure_ascii=False, allow_nan=False)
    try:
        from google.colab import files
        files.download(result_filename)
        if is_demo: files.download('pap-emulation-job.json')
    except ImportError:
        print('Saved locally:', result_filename, '(download manually outside Colab)')
    print('Result remains SIMULATED / UNVERIFIED. Import it through the PAP Colab Bridge.')


In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary. All rights reserved. AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary.
# SPDX-License-Identifier: GPL-3.0-only
%pip -q install 'numpy>=1.26,<3' 'scipy>=1.11,<2'

In [ ]:
import json, hashlib, platform, sys, time
import numpy as np
import scipy
from scipy.integrate import solve_ivp
print({'python':sys.version,'platform':platform.platform(),'numpy':np.__version__,'scipy':scipy.__version__})
PAP_USE_DEMO_JOB=True  # Set False to upload one browser-exported PAP job JSON
default_inputs={'A':[[-1.0]],'B':[[1.0]],'u':[1.0],'x0':[0.0],'C':[[1.0]],'D':[[0.0]],'times':np.linspace(0.0,5.0,51).tolist(),'method':'DOP853','rtol':1e-9,'atol':1e-11}
job,job_is_demo=pap_get_job('scipy.solve_ivp','scipy.linear_ode',default_inputs,PAP_USE_DEMO_JOB)
inputs=job['inputs']; A=np.asarray(inputs['A'],float); B=np.asarray(inputs['B'],float); u=np.asarray(inputs['u'],float); x0=np.asarray(inputs['x0'],float); Cmat=np.asarray(inputs['C'],float); D=np.asarray(inputs.get('D',np.zeros((Cmat.shape[0],B.shape[1]))),float); times=np.asarray(inputs['times'],float)
if A.ndim!=2 or A.shape[0]!=A.shape[1] or B.ndim!=2 or B.shape[0]!=A.shape[0] or u.shape!=(B.shape[1],) or x0.shape!=(A.shape[0],) or Cmat.ndim!=2 or Cmat.shape[1]!=A.shape[0] or D.shape!=(Cmat.shape[0],B.shape[1]): raise ValueError('Invalid state-space matrix dimensions')
if times.ndim!=1 or len(times)<2 or not np.isfinite(times).all() or np.any(np.diff(times)<=0): raise ValueError('times must be finite and strictly increasing')
rtol=float(inputs.get('rtol',1e-9)); atol=float(inputs.get('atol',1e-11)); print('Verified job',job['jobId'],'samples',len(times))

## 1. Execute solver comparison
The fixture solves `dx/dt = A x + B u`, `y = C x + D u`. For the default scalar fixture, the analytical solution is `x(t)=1-exp(-t)`. All six methods must converge and agree with the analytical reference within the stated threshold.

In [ ]:
def rhs(t,x): return A@x+B@u
methods=['RK23','RK45','DOP853','Radau','BDF','LSODA']; solver_results={}
for method in methods:
    sol=solve_ivp(rhs,(float(times[0]),float(times[-1])),x0,method=method,t_eval=times,rtol=rtol,atol=atol)
    if not sol.success or sol.y.shape!=(A.shape[0],len(times)) or not np.isfinite(sol.y).all(): raise RuntimeError(f'{method} failed: {sol.message}')
    y=(Cmat@sol.y+(D@u)[:,None]).T
    solver_results[method]={'nfev':int(sol.nfev),'njev':int(sol.njev),'nlu':int(sol.nlu),'termination':str(sol.message)}
    if A.shape==(1,1) and np.allclose(A,[[-1]]) and np.allclose(B,[[1]]) and np.allclose(u,[1]) and np.allclose(x0,[0]):
        err=np.abs(sol.y[0]-(1-np.exp(-(times-times[0])))); solver_results[method]['max_abs_error_analytical']=float(err.max()); assert err.max()<2e-8,(method,err.max())
solver_results

## 2. Export result
The result is bound to the exact uploaded job digest. Demo mode also downloads the matching job file so the result can be independently imported and checked by the browser bridge.

In [ ]:
method=inputs.get('method','DOP853'); chosen=solve_ivp(rhs,(float(times[0]),float(times[-1])),x0,method=method,t_eval=times,rtol=rtol,atol=atol)
y=(Cmat@chosen.y+(D@u)[:,None]).T
outputs=[{'sequence':int(i),'time':float(t),'values':{'state':chosen.y[:,i].astype(float).tolist(),'output':y[i].astype(float).tolist()}} for i,t in enumerate(times)]
result=pap_make_result(job,'scipy.solve_ivp',scipy.__version__,outputs,{'success':bool(chosen.success),'method':method,'rtol':rtol,'atol':atol,'nfev':int(chosen.nfev),'crossMethodValidation':solver_results},'cpu')
pap_export(job,result,job_is_demo)
print('Exported',len(outputs),'time samples; outcome remains simulated/unverified.')